# ==============================================================================
# 🏥 ACTUARIAL MODELING: INSURANCE CHARGES VIA ADVANCED ADABOOST REGRESSOR
# ==============================================================================
### Core Principles:
Medical claims feature heavy skewness and high-cost non-linear outlier surgeries.
Demonstrates systematic benchmarking of Drucker's 3 loss penalties (`linear`, `square`, `exponential`) wrapped in an enterprise `ColumnTransformer`.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Actuarial Environment ready.")


✅ STEP 1: Actuarial Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Medical Cost Personal dataset (1,338 policyholders).


In [2]:
# STEP 2 & 3: LOAD & HARMONIZE
data_path = 'data/insurance/insurance.csv'
df = pd.read_csv(data_path)

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]} policyholders, {df.shape[1]} attributes")
print(f"Annual Charges Summary:\n{df['charges'].describe().round(2)}")
df.head(3)


📊 Dataset Shape: 1338 policyholders, 7 attributes
Annual Charges Summary:
count     1338.00
mean     13270.42
std      12110.01
min       1121.87
25%       4740.29
50%       9382.03
75%      16639.91
max      63770.43
Name: charges, dtype: float64


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.90,0,yes,southwest,16884.9240
1,18,male,33.77,1,no,southeast,1725.5523
2,28,male,33.00,3,no,southeast,4449.4620


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `charges`.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['charges'])
y = df['charges']

print(f"Actuarial Features: {X.columns.tolist()}")
print(f"Missing attributes: {X.isnull().sum().sum()}")


Actuarial Features: ['age', 'sex', 'bmi', 'children', 'smoker', 'region']
Missing attributes: 0


In [4]:
# STEP 6: TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Cohort: {X_train.shape[0]}")
print(f"Test Cohort : {X_test.shape[0]}")


Train Cohort: 1070
Test Cohort : 268


## ⚙️ STEP 7: PREPROCESSOR PIPELINE
Encoding categorical policyholder attributes alongside standardized biometric indicators.


In [5]:
# STEP 7: PREPROCESSOR
cat_cols = ['sex', 'smoker', 'region']
num_cols = ['age', 'bmi', 'children']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
])
print("✅ STEP 7: Preprocessor ready.")


✅ STEP 7: Preprocessor ready.


## ⚖️ STEP 8: LOSS FUNCTION DUEL (Linear vs Square vs Exponential)
Empirically testing all three loss penalties in AdaBoost.R2.


In [6]:
# STEP 8: LOSS PENALTY DUEL
losses = ['linear', 'square', 'exponential']
loss_results = {}

for l in losses:
    pipe = Pipeline([
        ('prep', preprocessor),
        ('ada', AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=4), n_estimators=100, loss=l, random_state=42))
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    loss_results[l] = {
        'R2': r2_score(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'RMSE': np.sqrt(mean_squared_error(y_test, preds))
    }

print("🏆 LOSS FUNCTION BENCHMARK RESULTS:")
for l, res in loss_results.items():
    print(f"   Loss '{l:11s}' -> R²: {res['R2']*100:.2f}% | MAE: ${res['MAE']:,.2f} | RMSE: ${res['RMSE']:,.2f}")


🏆 LOSS FUNCTION BENCHMARK RESULTS:
   Loss 'linear     ' -> R²: 80.16% | MAE: $4,662.09 | RMSE: $5,550.37
   Loss 'square     ' -> R²: 57.83% | MAE: $7,593.98 | RMSE: $8,091.62
   Loss 'exponential' -> R²: 58.64% | MAE: $7,565.58 | RMSE: $8,012.76


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 5-FOLD CV
Optimizing `n_estimators`, `learning_rate`, and `loss`.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'ada__n_estimators': [50, 100, 150],
    'ada__learning_rate': [0.05, 0.1, 0.5],
    'ada__loss': ['linear', 'square']
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
pipe_tune = Pipeline([
    ('prep', preprocessor),
    ('ada', AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=4), random_state=42))
])

grid = GridSearchCV(pipe_tune, param_grid=param_grid, cv=cv, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Actuarial Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV R²: {grid.best_score_*100:.2f}%")


🏆 Best Actuarial Hyperparameters: {'ada__learning_rate': 0.05, 'ada__loss': 'linear', 'ada__n_estimators': 50}
🎯 Best 5-Fold CV R²: 84.53%


## 📊 STEP 10 & 11: ACTUARIAL REPORT & RESIDUALS
Evaluating underwriting error bounds.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📋 ACTUARIAL POLICY UNDERWRITING REPORT:")
print(f"   Mean Absolute Error (MAE) : ${mae:,.2f}")
print(f"   Root Mean Squared Error   : ${rmse:,.2f}")
print(f"   R² Coefficient            : {r2 * 100:.2f}%")


📋 ACTUARIAL POLICY UNDERWRITING REPORT:
   Mean Absolute Error (MAE) : $3,376.72
   Root Mean Squared Error   : $4,745.25
   R² Coefficient            : 85.50%


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & UNDERWRITING SMOKE TEST
Deploying pipeline and validating instant policy premium rating latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/insurance_adaboost_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_policy = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_premium = loaded_pipe.predict(sample_policy)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n💼 LIVE ACTUARIAL POLICY QUOTE:")
print(f"   Estimated Annual Premium: 💵 ${pred_premium:,.2f}")
print(f"   Actual Historical Charge: ${y_test.iloc[0]:,.2f}")
print(f"   Rating Latency          : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/insurance_adaboost_pipeline.joblib (38,254 bytes)

💼 LIVE ACTUARIAL POLICY QUOTE:
   Estimated Annual Premium: 💵 $12,793.79
   Actual Historical Charge: $9,095.07
   Rating Latency          : 40.725 ms (SLA < 2.0ms: CHECK)
